# TVAE — DGX training experiment

**Capstone:** Comparative evaluation of diffusion, GAN, and VAE models for synthetic clinical tabular data generation.

**Dataset:** Preprocessed UCI Diabetes 130-US Hospitals encounter records. This experiment trains a new TVAE on the fixed, patient-group-disjoint training split. The held-out test set is not used to fit the synthesizer.

**Protocol:** Fixed seed; explicit column-type metadata; 50 training epochs; one visible CUDA GPU; generated sample size equal to the training sample size. Outputs are written under new DGX-specific filenames.

## 1. Runtime and reproducibility

In [1]:
# Run this cell first, before any other code imports PyTorch.
import os
GPU_ID = 0  # Change only if a different GPU is assigned to you.
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

from pathlib import Path
import random
import json
import time
import numpy as np
import pandas as pd
import torch
import sdv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd().resolve()
if not (ROOT / 'data/processed/train_semantic.csv').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'data/processed/train_semantic.csv').is_file(), (
    'Cannot locate data/processed/train_semantic.csv. Open the project folder in VS Code.'
)

for folder in ['models', 'data/synthetic', 'results/training']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print('Project:', ROOT)
print('SDV:', getattr(sdv, '__version__', 'unknown'))
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'CUDA unavailable. Check that the DGX .venv kernel is selected.'


Project: /home/ayanasanthosh.ds25a/synthetic-clinical-data-generation
SDV: 1.38.5
PyTorch: 2.3.1+cu121
CUDA available: True
Visible GPU: NVIDIA A100-SXM4-80GB


## 2. Fixed training split and semantic schema

The eight encounter-count/duration variables are modeled as numerical; all remaining columns, including `readmitted_30`, are modeled as categorical. There are no patient identifiers in model inputs.

In [2]:
TRAIN_PATH = ROOT / 'data/processed/train_semantic.csv'
TEST_PATH = ROOT / 'data/processed/test_semantic.csv'
train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
TARGET = 'readmitted_30'
NUMERIC = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

assert TARGET in train.columns, f'Missing target: {TARGET}'
assert set(NUMERIC).issubset(train.columns), f'Missing numerical columns: {set(NUMERIC) - set(train.columns)}'
assert set(train.columns) == set(test.columns), 'Train and test schemas differ.'
assert not any(c in train.columns for c in ('patient_nbr', 'encounter_id')), 'Identifiers found in training data!'
assert not train.isna().any().any(), 'Training data contains missing values; verify preprocessing.'
assert not test.isna().any().any(), 'Held-out test contains missing values; verify preprocessing.'

# Read original integer-valued clinical counts as numeric; categorical codes as strings.
for col in train.columns:
    if col in NUMERIC:
        train[col] = pd.to_numeric(train[col], errors='raise')
    else:
        train[col] = train[col].astype(str)

train[TARGET] = train[TARGET].astype(str)
observed_labels = set(train[TARGET].unique())
assert observed_labels == {'0', '1'}, f'Unexpected binary target labels: {observed_labels}'

print('Training shape:', train.shape)
print('Held-out test shape:', test.shape)
print('Categorical columns:', len(train.columns) - len(NUMERIC))
print('Training target (%)')
print((train[TARGET].value_counts(normalize=True).sort_index() * 100).round(3))


Training shape: (79567, 45)
Held-out test shape: (19773, 45)
Categorical columns: 37
Training target (%)
readmitted_30
0    88.601
1    11.399
Name: proportion, dtype: float64


## 3. Metadata specification

Use an explicit schema rather than relying on automatic type inference. In particular, the binary readmission outcome is categorical and not a continuous numerical quantity.

In [3]:
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=train)

# Respect the data types established during preprocessing
for col in train.select_dtypes(include=["object", "category"]).columns:
    metadata.update_column(column_name=col, sdtype="categorical")

for col in train.select_dtypes(include=["number"]).columns:
    metadata.update_column(column_name=col, sdtype="numerical")

metadata.update_column(
    column_name="readmitted_30",
    sdtype="categorical"
)

metadata.validate()
metadata.validate_data(data=train)

print("Metadata validated successfully!")
print("SDV columns:", len(metadata.to_dict()["columns"]))

Metadata validated successfully!
SDV columns: 45


## 4. Fresh TVAE fitting

The experiment uses the entire training split, with **50 epochs** as the initial comparison setting. 

In [4]:
import time
import torch
from sdv.single_table import TVAESynthesizer

EPOCHS = 50
BATCH_SIZE = 500

print("CUDA available:", torch.cuda.is_available())

synthesizer = TVAESynthesizer(
    metadata=metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    enable_gpu=True,
    verbose=True
)

start = time.perf_counter()

synthesizer.fit(train)

training_time = time.perf_counter() - start

print(f"TVAE training completed!")
print(f"Training time: {training_time / 60:.2f} minutes")

CUDA available: True


/home/ayanasanthosh.ds25a/synthetic-clinical-data-generation/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/ayanasanthosh.ds25a/synthetic-clinical-data-generation/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: +34.24: 100%|██████████| 50/50 [02:19<00:00,  2.78s/it]

TVAE training completed!
Training time: 3.24 minutes


In [5]:
from pathlib import Path

# Find project folder
root = Path.cwd()
if not (root / "data" / "processed").exists():
    root = root.parent

# Create output directories
(root / "models").mkdir(exist_ok=True)
(root / "data" / "synthetic").mkdir(exist_ok=True)
(root / "results" / "training").mkdir(parents=True, exist_ok=True)

# Save trained model
synthesizer.save(
    filepath=str(root / "models" / "tvae_50ep.pkl")
)

# Save metadata for reproducibility
metadata.save_to_json(
    filepath=str(root / "results" / "training" / "tvae_metadata.json")
)

# Generate same number of rows as training dataset
synthetic_tvae = synthesizer.sample(num_rows=len(train))

# Save synthetic dataset
synthetic_tvae.to_csv(
    root / "data" / "synthetic" / "tvae_50ep.csv",
    index=False
)

print("Synthetic shape:", synthetic_tvae.shape)

print("\nReal target distribution:")
print(train["readmitted_30"].value_counts(dropna=False))

print("\nSynthetic target distribution:")
print(synthetic_tvae["readmitted_30"].value_counts(dropna=False))

print("\nTVAE model and synthetic dataset saved!")

Synthetic shape: (79567, 45)

Real target distribution:
readmitted_30
0    70497
1     9070
Name: count, dtype: int64

Synthetic target distribution:
readmitted_30
0    79567
Name: count, dtype: int64

TVAE model and synthetic dataset saved!


## . Interpretation and next experiments

Inspect the 30-day readmission prevalence and numerical feature ranges before proceeding to model utility assessment. If the TVAE samples only the majority class, TSTR cannot be meaningfully computed for the positive class; retain this result as evidence of class-collapse rather than rewriting labels.

The CTGAN and TabDDPM experiments will use the identical training split and held-out evaluation protocol. The held-out test data must not be included in any synthesizer training.